# PriceFusion V0.6 — Transformer Text Embeddings

**Project:** PriceFusion — Multimodal Product Price Intelligence  
**Stage:** V0.6  
**Goal:** Add dense semantic text representations and test whether transformer embeddings improve the current V0.5 hybrid benchmark.

### Experiments
- Transformer embedding + Ridge (`log1p(price)`)
- Transformer embedding + LightGBM (`log1p(price)`)
- Transformer + V0.4.1 structured features + LightGBM
- Ensemble with the current V0.5 best model

### Current benchmark to beat
**V0.5 Text + LightGBM Ensemble — SMAPE 52.3737%**

Embeddings are cached under `artifacts/embeddings/v06/`, so subsequent notebook runs do not recompute them.

## 1. Imports

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

RANDOM_STATE = 42
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
V05_BEST_SMAPE = 52.373719

print('V0.6 environment ready.')

V0.6 environment ready.


## 2. Resolve Project Paths

In [2]:
cwd = Path.cwd().resolve()

if (cwd / 'dataset').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'dataset').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not find PriceFusion project root.')

DATASET_DIR = PROJECT_ROOT / 'dataset'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FEATURES_DIR = ARTIFACTS_DIR / 'features'
EMBEDDINGS_DIR = ARTIFACTS_DIR / 'embeddings' / 'v06'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models' / 'v06_transformer'

for p in [EMBEDDINGS_DIR, METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATASET_DIR / 'train.csv'
TEST_CSV = DATASET_DIR / 'test.csv'
TRAIN_FEATURES = FEATURES_DIR / 'v04_train_features.parquet'
VALID_FEATURES = FEATURES_DIR / 'v04_validation_features.parquet'
TEST_FEATURES = FEATURES_DIR / 'v04_test_features.parquet'
V05_PREDS = PREDICTIONS_DIR / 'v05_validation_predictions.csv'

print('Project root:', PROJECT_ROOT)
print('Embedding cache:', EMBEDDINGS_DIR)

Project root: /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion
Embedding cache: /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion/artifacts/embeddings/v06


## 3. Detect Compute Device

In [3]:
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

BATCH_SIZE = 128 if DEVICE == 'cuda' else (64 if DEVICE == 'mps' else 32)

print('Device:', DEVICE)
print('Embedding batch size:', BATCH_SIZE)

Device: mps
Embedding batch size: 64


## 4. Load Raw Text and Structured Features

In [4]:
raw_train = pd.read_csv(TRAIN_CSV)
raw_test = pd.read_csv(TEST_CSV)

train_struct = pd.read_parquet(TRAIN_FEATURES)
valid_struct = pd.read_parquet(VALID_FEATURES)
test_struct = pd.read_parquet(TEST_FEATURES)

print('Structured train:', train_struct.shape)
print('Structured valid:', valid_struct.shape)
print('Structured test :', test_struct.shape)

raw_text_map = raw_train.set_index('sample_id')['catalog_content']
test_text_map = raw_test.set_index('sample_id')['catalog_content']

train_texts = train_struct['sample_id'].map(raw_text_map).fillna('').astype(str).tolist()
valid_texts = valid_struct['sample_id'].map(raw_text_map).fillna('').astype(str).tolist()
test_texts = test_struct['sample_id'].map(test_text_map).fillna('').astype(str).tolist()

assert all(len(x) > 0 for x in train_texts)
assert all(len(x) > 0 for x in valid_texts)
assert all(len(x) > 0 for x in test_texts)

print('Text alignment complete.')

Structured train: (60000, 59)
Structured valid: (15000, 59)
Structured test : (75000, 58)
Text alignment complete.


## 5. Load Sentence Transformer

`all-MiniLM-L6-v2` produces compact 384-dimensional sentence embeddings. Long catalog entries are automatically truncated to the model's supported sequence length.

In [5]:
encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)

print('Model:', MODEL_NAME)
print('Max sequence length:', encoder.max_seq_length)
print('Embedding dimension:', encoder.get_sentence_embedding_dimension())

Model: sentence-transformers/all-MiniLM-L6-v2
Max sequence length: 256
Embedding dimension: 384


## 6. Cached Embedding Helper

In [6]:
def encode_or_load(texts, ids, split_name):
    emb_path = EMBEDDINGS_DIR / f'{split_name}_embeddings.npy'
    ids_path = EMBEDDINGS_DIR / f'{split_name}_ids.npy'

    ids_arr = np.asarray(ids)

    if emb_path.exists() and ids_path.exists():
        saved_ids = np.load(ids_path)
        if np.array_equal(saved_ids, ids_arr):
            embeddings = np.load(emb_path)
            print(f'Loaded cached {split_name}:', embeddings.shape)
            return embeddings
        else:
            print(f'Cached {split_name} IDs do not match; recomputing.')

    embeddings = encoder.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    np.save(emb_path, embeddings)
    np.save(ids_path, ids_arr)
    print(f'Saved {split_name}:', embeddings.shape)
    return embeddings

## 7. Generate / Load Transformer Embeddings

**Note:** The first run is the expensive part. Once these three arrays are saved, future runs load them directly.

In [7]:
E_train = encode_or_load(train_texts, train_struct['sample_id'].values, 'train')
E_valid = encode_or_load(valid_texts, valid_struct['sample_id'].values, 'validation')
E_test = encode_or_load(test_texts, test_struct['sample_id'].values, 'test')

assert E_train.shape[0] == len(train_struct)
assert E_valid.shape[0] == len(valid_struct)
assert E_test.shape[0] == len(test_struct)
assert E_train.shape[1] == E_valid.shape[1] == E_test.shape[1]

print('Train embeddings:', E_train.shape)
print('Valid embeddings:', E_valid.shape)
print('Test embeddings :', E_test.shape)

Loaded cached train: (60000, 384)
Loaded cached validation: (15000, 384)
Loaded cached test: (75000, 384)
Train embeddings: (60000, 384)
Valid embeddings: (15000, 384)
Test embeddings : (75000, 384)


## 8. Evaluation Helpers

In [8]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    num = np.abs(y_true - y_pred)
    ratio = np.divide(num, denom, out=np.zeros_like(num), where=denom != 0)
    return float(np.mean(ratio) * 100.0)

def clip_positive(pred):
    return np.maximum(np.asarray(pred, dtype=float), 0.01)

def evaluate(y_true, pred, name):
    pred = clip_positive(pred)
    return {
        'model': name,
        'smape': smape(y_true, pred),
        'mae': mean_absolute_error(y_true, pred),
        'median_ae': median_absolute_error(y_true, pred),
        'min_prediction': float(pred.min()),
        'max_prediction': float(pred.max()),
    }

def show(m):
    print(m['model'])
    print(f"SMAPE     : {m['smape']:.4f}%")
    print(f"MAE       : {m['mae']:.4f}")
    print(f"Median AE : {m['median_ae']:.4f}")

y_train = train_struct['price'].values
y_valid = valid_struct['price'].values

results = []
predictions = {}

## 9. Transformer Embeddings + Ridge

In [9]:
ridge = Ridge(alpha=10.0)
ridge.fit(E_train, np.log1p(y_train))

ridge_pred = np.expm1(ridge.predict(E_valid))
ridge_pred = clip_positive(ridge_pred)

m = evaluate(y_valid, ridge_pred, 'Transformer Embeddings + Ridge')
results.append(m)
predictions['transformer_ridge'] = ridge_pred
show(m)

Transformer Embeddings + Ridge
SMAPE     : 63.7577%
MAE       : 14.4848
Median AE : 7.3453


## 10. Transformer Embeddings + LightGBM

In [ ]:
emb_feature_names = [f'emb_{i:03d}' for i in range(E_train.shape[1])]

E_train_df = pd.DataFrame(E_train, columns=emb_feature_names)
E_valid_df = pd.DataFrame(E_valid, columns=emb_feature_names)

lgb_emb = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=3000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgb_emb.fit(
    E_train_df,
    np.log1p(y_train),
    eval_set=[(E_valid_df, np.log1p(y_valid))],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

emb_lgb_pred = np.expm1(lgb_emb.predict(E_valid_df, num_iteration=lgb_emb.best_iteration_))
emb_lgb_pred = clip_positive(emb_lgb_pred)

m = evaluate(y_valid, emb_lgb_pred, 'Transformer Embeddings + LightGBM')
results.append(m)
predictions['transformer_lgb'] = emb_lgb_pred
show(m)

## 11. Prepare Transformer + Structured Features

In [ ]:
TARGET = 'price'
ID_COL = 'sample_id'
CAT_COL = 'unit_normalized'

structured_cols = [c for c in train_struct.columns if c not in {TARGET, ID_COL}]
numeric_struct_cols = [c for c in structured_cols if c != CAT_COL]

S_train = train_struct[numeric_struct_cols].copy()
S_valid = valid_struct[numeric_struct_cols].copy()

for col in numeric_struct_cols:
    S_train[col] = pd.to_numeric(S_train[col], errors='coerce')
    S_valid[col] = pd.to_numeric(S_valid[col], errors='coerce')

# Factorize the single categorical unit column using a shared vocabulary.
all_units = pd.concat([
    train_struct[CAT_COL],
    valid_struct[CAT_COL],
    test_struct[CAT_COL]
], ignore_index=True).fillna('unknown').astype(str)
unit_categories = sorted(all_units.unique().tolist())
unit_map = {u: i for i, u in enumerate(unit_categories)}

S_train['unit_code'] = train_struct[CAT_COL].fillna('unknown').astype(str).map(unit_map).astype(int)
S_valid['unit_code'] = valid_struct[CAT_COL].fillna('unknown').astype(str).map(unit_map).astype(int)

S_train = S_train.reset_index(drop=True)
S_valid = S_valid.reset_index(drop=True)

combined_train = pd.concat([S_train, E_train_df], axis=1)
combined_valid = pd.concat([S_valid, E_valid_df], axis=1)

print('Combined train shape:', combined_train.shape)
print('Combined valid shape:', combined_valid.shape)

## 12. Transformer + Structured LightGBM

In [ ]:
lgb_hybrid = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=3500,
    learning_rate=0.018,
    num_leaves=63,
    min_child_samples=25,
    subsample=0.9,
    colsample_bytree=0.75,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgb_hybrid.fit(
    combined_train,
    np.log1p(y_train),
    eval_set=[(combined_valid, np.log1p(y_valid))],
    callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
)

hybrid_pred = np.expm1(
    lgb_hybrid.predict(combined_valid, num_iteration=lgb_hybrid.best_iteration_)
)
hybrid_pred = clip_positive(hybrid_pred)

m = evaluate(y_valid, hybrid_pred, 'Transformer + Structured LightGBM')
results.append(m)
predictions['transformer_structured_lgb'] = hybrid_pred
show(m)

## 13. Load V0.5 Best Validation Prediction

In [ ]:
assert V05_PREDS.exists(), 'Missing artifacts/predictions/v05_validation_predictions.csv'

v05 = pd.read_csv(V05_PREDS)
v05_cols = ['sample_id', 'actual_price', 'text_lgb_ensemble']
v05 = v05[v05_cols].copy()

aligned = valid_struct[['sample_id', 'price']].merge(v05, on='sample_id', how='left')
assert aligned['text_lgb_ensemble'].notna().all()
assert np.allclose(aligned['price'], aligned['actual_price'])

v05_pred = aligned['text_lgb_ensemble'].values

m = evaluate(y_valid, v05_pred, 'V0.5 Text + LightGBM Ensemble')
results.append(m)
predictions['v05_best'] = v05_pred
show(m)

## 14. Optimize V0.5 + Transformer Hybrid Ensemble

In [ ]:
blend_rows = []

for v05_weight in np.linspace(0.0, 1.0, 41):
    transformer_weight = 1.0 - v05_weight
    p = v05_weight * v05_pred + transformer_weight * hybrid_pred
    blend_rows.append({
        'v05_weight': float(v05_weight),
        'transformer_weight': float(transformer_weight),
        'smape': smape(y_valid, p),
    })

blend_df = pd.DataFrame(blend_rows).sort_values('smape').reset_index(drop=True)
display(blend_df.head(12))

best_v05_weight = float(blend_df.loc[0, 'v05_weight'])
best_transformer_weight = float(blend_df.loc[0, 'transformer_weight'])

final_blend_pred = clip_positive(
    best_v05_weight * v05_pred + best_transformer_weight * hybrid_pred
)

m = evaluate(y_valid, final_blend_pred, 'V0.5 + Transformer Hybrid Ensemble')
results.append(m)
predictions['v06_final_ensemble'] = final_blend_pred
show(m)

## 15. V0.6 Leaderboard

In [ ]:
leaderboard = pd.DataFrame(results).sort_values('smape').reset_index(drop=True)
display(leaderboard)
leaderboard.to_csv(METRICS_DIR / 'v06_transformer_leaderboard.csv', index=False)

## 16. Bucket-Level Analysis

In [ ]:
PRICE_BINS = [0, 5, 10, 25, 50, 100, 250, np.inf]
PRICE_LABELS = ['$0–5', '$5–10', '$10–25', '$25–50', '$50–100', '$100–250', '$250+']
buckets = pd.cut(y_valid, bins=PRICE_BINS, labels=PRICE_LABELS, right=False, include_lowest=True)

bucket_rows = []
for key, pred in predictions.items():
    tmp = pd.DataFrame({'actual': y_valid, 'pred': pred, 'bucket': buckets.astype(str)})
    for label in PRICE_LABELS:
        part = tmp[tmp['bucket'] == label]
        if len(part) == 0:
            continue
        bucket_rows.append({
            'model': key,
            'price_bucket': label,
            'rows': len(part),
            'smape': smape(part['actual'], part['pred']),
            'mae': mean_absolute_error(part['actual'], part['pred']),
        })

bucket_df = pd.DataFrame(bucket_rows)
display(bucket_df)
bucket_df.to_csv(METRICS_DIR / 'v06_bucket_metrics.csv', index=False)

## 17. Hybrid Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': combined_train.columns,
    'importance': lgb_hybrid.feature_importances_,
}).sort_values('importance', ascending=False)

display(importance.head(30))
importance.to_csv(METRICS_DIR / 'v06_hybrid_feature_importance.csv', index=False)

top = importance.head(20)
plt.figure(figsize=(10, 7))
plt.barh(top['feature'][::-1], top['importance'][::-1])
plt.title('V0.6 Transformer + Structured Feature Importance')
plt.xlabel('LightGBM importance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'v06_hybrid_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. Save Validation Predictions

In [ ]:
pred_df = valid_struct[['sample_id', 'price']].copy().rename(columns={'price': 'actual_price'})
for key, pred in predictions.items():
    pred_df[key] = pred

pred_df.to_csv(PREDICTIONS_DIR / 'v06_validation_predictions.csv', index=False)
display(pred_df.head())

## 19. Save Models and Configuration

In [ ]:
joblib.dump(ridge, MODELS_DIR / 'transformer_ridge.joblib')
joblib.dump(lgb_emb, MODELS_DIR / 'transformer_lightgbm.joblib')
joblib.dump(lgb_hybrid, MODELS_DIR / 'transformer_structured_lightgbm.joblib')

config = {
    'version': 'V0.6',
    'encoder_model': MODEL_NAME,
    'embedding_dimension': int(E_train.shape[1]),
    'max_sequence_length': int(encoder.max_seq_length),
    'normalized_embeddings': True,
    'v05_weight': best_v05_weight,
    'transformer_hybrid_weight': best_transformer_weight,
    'best_model': str(leaderboard.loc[0, 'model']),
    'best_smape': float(leaderboard.loc[0, 'smape']),
}

with (MODELS_DIR / 'v06_config.json').open('w') as f:
    json.dump(config, f, indent=2)

print('Saved V0.6 models and configuration.')

## 20. Final V0.6 Summary

In [ ]:
best_smape = float(leaderboard.loc[0, 'smape'])

print('=' * 92)
print('PRICEFUSION V0.6 — TRANSFORMER TEXT EMBEDDINGS COMPLETE')
print('=' * 92)
print(f'Encoder          : {MODEL_NAME}')
print(f'Device           : {DEVICE}')
print(f'Embedding dim    : {E_train.shape[1]}')
print(f'Train rows       : {len(train_struct):,}')
print(f'Validation rows  : {len(valid_struct):,}')
print('-' * 92)
for _, row in leaderboard.iterrows():
    print(f"{row['model']:<52} SMAPE={row['smape']:.4f}%   MAE={row['mae']:.4f}")
print('-' * 92)
print(f"BEST MODEL       : {leaderboard.loc[0, 'model']}")
print(f"BEST SMAPE       : {best_smape:.4f}%")
print(f"V0.5 BEST        : {V05_BEST_SMAPE:.4f}%")
print(f"GAIN VS V0.5     : {V05_BEST_SMAPE - best_smape:.4f} SMAPE points")
print('=' * 92)
print('Next milestone: PriceFusion V0.7 — Vision / Image Embeddings')